# 12: Which generator made it? (E4)

Up to now every detector answered a yes/no question: real or generated. Attribution asks more: **which of the seven sources** (Real, ADM, GLIDE, Midjourney, SD15, VQDM, Wukong) produced the image. This is source identification, the generator version of identifying which camera took a photo.

Two attribution models:
- **A1: CLIP features + logistic regression** (7 classes). Reuses the CLIP features saved by notebook 06, so it trains in about a minute.
- **A2: ResNet-50 fine-tuned for 7 classes.** Same trainer as M1, with seven outputs instead of one.

Every class has the same number of images in train, validation and test (real images are sub-sampled), so guessing gives 1/7 = 14.3%.

GPU runtime needed for A2 (about 10 minutes).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json
import shutil
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, f1_score, log_loss
from torch.utils.data import DataLoader, Dataset
from torchvision.models import ResNet50_Weights, resnet50

ROOT = Path("/content/drive/MyDrive/minor-project")
LOCAL = Path("/content/processed")
RESULTS = ROOT / "results"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## Settings

In [ ]:
CLASSES = ["Real", "ADM", "GLIDE", "Midjourney", "SD15", "VQDM", "Wukong"]
CLIP_TAG = "clip-vit-large-patch14"
RUN_NAME = "e4_a2_resnet50"
EPOCHS = 5
BATCH_SIZE = 64
LEARNING_RATE = 1e-4
NUM_WORKERS = 2
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

## Data

Real images outnumber each generator about six to one, so in every split we keep a random sample of real images equal to the size of one generator class.

In [ ]:
LOCAL.mkdir(exist_ok=True)
for split in ["train", "val", "test"]:
    for suffix in ["_images.npy", "_meta.csv"]:
        name = split + suffix
        if not (LOCAL / name).exists():
            shutil.copy(ROOT / "processed" / name, LOCAL / name)


def load_split(split):
    images = np.load(LOCAL / f"{split}_images.npy", mmap_mode="r")
    meta = pd.read_csv(LOCAL / f"{split}_meta.csv")
    per_class = meta[meta["generator"] != "Real"]["generator"].value_counts().max()
    reals = meta[meta["generator"] == "Real"].sample(n=per_class, random_state=SEED)
    meta = pd.concat([reals, meta[meta["generator"] != "Real"]])
    meta["y"] = meta["generator"].map(CLASSES.index)
    return images, meta


data = {split: load_split(split) for split in ["train", "val", "test"]}
for split, (_, meta) in data.items():
    print(split, meta["generator"].value_counts().reindex(CLASSES).to_dict())

In [ ]:
def report(name, meta, probs):
    pred = probs.argmax(1)
    y = meta["y"].to_numpy()
    result = {
        "accuracy": float((pred == y).mean()),
        "macro_f1": float(f1_score(y, pred, average="macro")),
        "per_class_recall": {c: float((pred[y == k] == k).mean()) for k, c in enumerate(CLASSES)},
    }
    out = meta[["hf_split", "hf_index", "generator", "y"]].copy()
    out["predicted"] = [CLASSES[k] for k in pred]
    for k, c in enumerate(CLASSES):
        out[f"p_{c}"] = probs[:, k]
    out.to_csv(RESULTS / f"{name}_test_predictions.csv", index=False)
    print(f"{name}: accuracy {result['accuracy']:.3f}, macro-F1 {result['macro_f1']:.3f}")
    return result, confusion_matrix(y, pred, labels=range(len(CLASSES)), normalize="true")

## A1: CLIP features + logistic regression

The regularisation strength C is chosen on the validation set, as in notebook 06.

In [ ]:
feats = {split: np.load(ROOT / "features" / f"{CLIP_TAG}_{split}.npy") for split in data}
X = {split: feats[split][meta.index] for split, (_, meta) in data.items()}
Y = {split: meta["y"].to_numpy() for split, (_, meta) in data.items()}

search = []
for C in [0.1, 1, 10, 100]:
    clf = LogisticRegression(C=C, max_iter=5000).fit(X["train"], Y["train"])
    val_probs = clf.predict_proba(X["val"])
    search.append({"C": C, "val_loss": float(log_loss(Y["val"], val_probs)),
                   "val_accuracy": float((val_probs.argmax(1) == Y["val"]).mean())})
    print(f"C={C:<5} val loss {search[-1]['val_loss']:.4f} | val acc {search[-1]['val_accuracy']:.3f}")

best_C = min(search, key=lambda r: r["val_loss"])["C"]
clf = LogisticRegression(C=best_C, max_iter=5000).fit(X["train"], Y["train"])
a1, a1_cm = report("e4_a1_clip", data["test"][1], clf.predict_proba(X["test"]))
a1["best_C"] = best_C

## A2: ResNet-50 fine-tuned for 7 classes

In [ ]:
class ImageArrayDataset(Dataset):
    def __init__(self, images, meta):
        self.images = images
        self.rows = meta.index.to_numpy()
        self.labels = meta["y"].to_numpy()

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        return torch.from_numpy(np.array(self.images[self.rows[i]])), self.labels[i]


sets = {split: ImageArrayDataset(*data[split]) for split in data}
loaders = {
    split: DataLoader(sets[split], batch_size=BATCH_SIZE if split == "train" else BATCH_SIZE * 2,
                      shuffle=split == "train", num_workers=NUM_WORKERS, pin_memory=True)
    for split in data
}

MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)


def to_model_input(batch, augment=False):
    x = batch.to(device, non_blocking=True).permute(0, 3, 1, 2).float() / 255
    if augment:
        flip = torch.rand(x.shape[0], device=device) < 0.5
        x[flip] = x[flip].flip(-1)
    return (x - MEAN) / STD


model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
model.fc = nn.Linear(model.fc.in_features, len(CLASSES))
model = model.to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler(device.type, enabled=use_amp)


@torch.no_grad()
def predict(loader):
    model.eval()
    probs = []
    for images, _ in loader:
        with torch.autocast(device.type, enabled=use_amp):
            logits = model(to_model_input(images))
        probs.append(torch.softmax(logits.float(), dim=1).cpu())
    return torch.cat(probs).numpy()

In [ ]:
CKPT = ROOT / "checkpoints" / f"{RUN_NAME}.pt"
CKPT.parent.mkdir(exist_ok=True)
history = []
best_val_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    model.train()
    start = time.time()
    running_loss = 0.0
    for images, labels in loaders["train"]:
        x = to_model_input(images, augment=True)
        y = labels.to(device)
        with torch.autocast(device.type, enabled=use_amp):
            loss = loss_fn(model(x), y)
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * len(y)

    val_probs = predict(loaders["val"])
    val_loss = float(log_loss(sets["val"].labels, val_probs, labels=range(len(CLASSES))))
    val_acc = float((val_probs.argmax(1) == sets["val"].labels).mean())
    history.append({"epoch": epoch, "train_loss": running_loss / len(sets["train"]), "val_loss": val_loss, "val_accuracy": val_acc})
    print(f"Epoch {epoch}: train loss {history[-1]['train_loss']:.4f} | val loss {val_loss:.4f} | val acc {val_acc:.3f} | {time.time() - start:.0f}s")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), CKPT)
        print("  saved as best so far")

model.load_state_dict(torch.load(CKPT, map_location=device))
a2, a2_cm = report("e4_a2_resnet50", data["test"][1], predict(loaders["test"]))
a2["history"] = history

## Results

In [ ]:
with open(RESULTS / "e4_attribution.json", "w") as f:
    json.dump({"A1 CLIP + logistic regression": a1, "A2 ResNet-50": a2, "clip_search": search}, f, indent=2)

recall = pd.DataFrame({"A1 CLIP": a1["per_class_recall"], "A2 ResNet-50": a2["per_class_recall"]}).T
recall["accuracy"] = [a1["accuracy"], a2["accuracy"]]
recall["macro-F1"] = [a1["macro_f1"], a2["macro_f1"]]
print("Per-class recall (fraction of each source's test images attributed correctly):")
display(recall.round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6.4))
for ax, (title, cm, result) in zip(axes, [("A1: CLIP + logistic regression", a1_cm, a1), ("A2: ResNet-50", a2_cm, a2)]):
    ax.imshow(cm * 100, cmap="Blues", vmin=0, vmax=100)
    for i in range(len(CLASSES)):
        for j in range(len(CLASSES)):
            value = cm[i, j] * 100
            if value >= 0.5:
                ax.text(j, i, f"{value:.0f}", ha="center", va="center", fontsize=9,
                        color="white" if value > 55 else "#333333")
    ax.set_xticks(range(len(CLASSES)), CLASSES, rotation=35, ha="right")
    ax.set_yticks(range(len(CLASSES)), CLASSES)
    ax.set_xlabel("predicted source")
    ax.set_title(f"{title}\naccuracy {result['accuracy']:.1%}", fontsize=11)
axes[0].set_ylabel("true source")
plt.tight_layout()
plt.savefig(RESULTS / "e4_confusion.png", dpi=120)
plt.show()